In [3]:
import pandas as pd 
from glob import glob 
root_dir = '/home/work/yuna/HPA/evaluation/scored'

model_names = [
            "OpenGVLab/InternVL3_5-8B",
            "OpenGVLab/InternVL3_5-2B",
            "OpenGVLab/InternVL3_5-4B",
            "OpenGVLab/InternVL3_5-1B",

            "Qwen/Qwen3-VL-2B-Instruct", 
            "Qwen/Qwen3-VL-4B-Instruct", 
            "Qwen/Qwen3-VL-8B-Instruct",

            "llava-hf/llava-1.5-7b-hf", 
            "llava-hf/llava-v1.6-vicuna-7b-hf", 
            "llava-hf/llava-v1.6-mistral-7b-hf", 

            "Qwen/Qwen3-8B-Base", 
            "Qwen/Qwen3-4B-Base", 
            "Qwen/Qwen3-1.7B-Base" , 
            "Qwen/Qwen3-0.6B-Base" # doesnt work 
            "Qwen/Qwen3-0.6B", 
            "Qwen/Qwen3-8B", 
            "Qwen/Qwen3-4B", 
]

def find_matching(f, targets): 
    for t in targets : 
        if t in f : 
            f = f.replace(f'{t}', '')   
            return t, f 
    print(f"cannot find matching {f} in {targets}") 

def get_summary(dataset='mmstar'): 
    # df = df.groupby(['model', 'folder', 'condition'])['correct'].mean()

    files = glob(f"{root_dir}/*/*{dataset}*.jsonl")  + glob(f"{root_dir}/*/*/*/{dataset}*.jsonl")

    dfs= []
    for f in files: 
        try: 
            df = pd.read_json(f, lines=True)
            if 'finetuned' in f : 
                df['model'] = f.split('/')[-2].replace('fold_0', '')
            else: 
                df['model'], f = find_matching(f, [model.split('/')[-1] for model in model_names])  
            df['condition'] = f.split('/')[-1][:-6].replace(f'_', ' ').replace('vqa 1k', '').replace(f'{dataset}', '').strip()
            dfs.append(df)
        except Exception as e: 
            print(e)
    df = pd.concat(dfs)
    print(len(files) ) 

    pt = df.pivot_table(
        index=['model'],  
        columns=['condition'], 
        values=['correct'],
        aggfunc=['mean', 'count']
    )
    pt = pt.round(4)
    pt.to_csv(f"./summary_{dataset}.csv")
    return df , pt 

In [15]:
!python /home/work/yuna/HPA/evaluation/score_humans.py --human_data_dir n20 


📊 Processing Raw Human Responses
   Session: s1
   Data dir: /home/work/yuna/HPA/data/humans/n20

📚 Loading annotations...
Length of MMStar questions: 267

Processing VQA (text) responses...
📋 ANSWER PREPROCESSING PIPELINE

[1/5] Loading data...
✓ Loaded 641 questions from /home/work/yuna/HPA/dataset/questions/s1.csv
/home/work/yuna/HPA/data/humans/n20/1ed1a464_20251204_110002/answers.csv is incomplete, skip
/home/work/yuna/HPA/data/humans/n20/9d6d8564_20251210_233650/answers.csv is incomplete, skip
/home/work/yuna/HPA/data/humans/n20/74d408e5_20251213_134035/answers.csv is incomplete, skip
/home/work/yuna/HPA/data/humans/n20/99f271ae_20251203_111155/answers.csv is incomplete, skip
/home/work/yuna/HPA/data/humans/n20/ba2d2124_20251208_223639/answers.csv is incomplete, skip
✓ Loaded 12828 responses from 20 files

[2/5] Translating Korean answers...
✓ Loaded 1381 cached translations from /home/work/yuna/HPA/preprocessing/translation_cache.json

🌐 Translation Status:
   Total responses: 

In [ ]:
!python /home/work/yuna/HPA/evaluation/score_results.py  --input_dir finetuned  
!python /home/work/yuna/HPA/evaluation/score_results.py  --input_dir pretrained  


📊 Processing Raw Human Responses
   Session: s1
   Data dir: /home/work/yuna/HPA/data/humans/n20

📚 Loading annotations...
manually matched 802
manually matched 287
manually matched 588
manually matched 214
Length of MMStar questions: 267

Processing VQA (text) responses...
📋 ANSWER PREPROCESSING PIPELINE

[1/5] Loading data...
✓ Loaded 641 questions from /home/work/yuna/HPA/dataset/questions/s1.csv
/home/work/yuna/HPA/data/humans/n20/1ed1a464_20251204_110002/answers.csv is incomplete, skip
/home/work/yuna/HPA/data/humans/n20/9d6d8564_20251210_233650/answers.csv is incomplete, skip
/home/work/yuna/HPA/data/humans/n20/74d408e5_20251213_134035/answers.csv is incomplete, skip
/home/work/yuna/HPA/data/humans/n20/99f271ae_20251203_111155/answers.csv is incomplete, skip
/home/work/yuna/HPA/data/humans/n20/ba2d2124_20251208_223639/answers.csv is incomplete, skip
✓ Loaded 12828 responses from 20 files

[2/5] Translating Korean answers...
✓ Loaded 1381 cached translations from /home/work/yuna/

In [11]:
model_results = {}
for ds in ['mmstar', 'spubench', 'vqa_5k', 'vqa_1k']: 
    model_results[ds], pv = get_summary(ds) 

70
70
68
61


In [13]:
human_mc=pd.read_csv('/home/work/yuna/HPA/evaluation/scored/humans/human_mc_per_question.csv')
human_mc['model'] = "humans"  
human_mc.rename(columns={'mean_accuracy': 'correct'}, inplace=True) 
human_mc['condition'] = "inst blind" 
human_mc # .groupby('category').agg({'mean': 'correct', 'count': "correct"}) 

,Unnamed: 0,model,condition


In [65]:
qids = human_mc.pid.unique() 
model_mc = model_results['mmstar']
human_mc['pid'] = human_mc['pid'].astype('Int64')
model_mc['pid'] = model_mc['pid'].astype('Int64') 
print(len(qids))
mmstar_human_comparison = pd.concat([model_mc[model_mc['pid'].isin(qids)] , human_mc])
len(mmstar_human_comparison)
pt = mmstar_human_comparison.pivot_table( 
    index=['model'], 
    columns=['condition'], 
    values=['correct'],
    aggfunc=['mean', 'count']
)
pt = pt.round(4)
pt.to_csv(f'./mmstar_human_comparison.csv')
pt 

247


mean                       \
                                                correct                        
condition                                                   blind inst blind   
model                                                                          
InternVL3_5-1B                                  0.45749  0.291498   0.279352   
InternVL3_5-2B                                 0.538462  0.255061   0.234818   
InternVL3_5-4B                                 0.651822  0.271255   0.246964   
InternVL3_5-8B                                 0.650407  0.323887   0.327935   
InternVL3_5-8B_A1_vqa_gt                       0.700405       NaN   0.323887   
Qwen3-8B-Base                                  0.174089       NaN        NaN   
Qwen3-VL-2B-Instruct                           0.587045  0.218623   0.271255   
Qwen3-VL-4B-Instruct                           0.663968  0.299595   0.283401   
Qwen3-VL-4B-Instruct_A1_vqa_gt                 0.635628       NaN   0.271255   
Qwen3-VL-4B-Instruct_A2_vqa_10_blind_inst      0.631579       NaN   0.287449   
Qwen3-VL-4B-Instruct_A3_vqa_15_blind_inst      0.631579       NaN   0.315789   
Qwen3-VL-4B-Instruct_A4_mmstar_15_blind_inst   0.639676       NaN   0.299595   
Qwen3-VL-4B-Instruct_SFT_vqa_15_blind_inst     0.623482       NaN   0.287449   
Qwen3-VL-4B-Instruct_SFT_vqa_gt                0.635628       NaN   0.295547   
Qwen3-VL-8B-Instruct                           0.684211  0.291498   0.323887   
Qwen3-VL-8B-Instruct_A1_vqa_gt                 0.696356       NaN   0.299595   
Qwen3-VL-8B-Instruct_A2_vqa_10_blind_inst      0.692308       NaN   0.287449   
Qwen3-VL-8B-Instruct_A3_vqa_15_blind_inst      0.696356       NaN   0.299595   
Qwen3-VL-8B-Instruct_A4_mmstar_15_blind_inst   0.696356       NaN   0.319838   
Qwen3-VL-8B-Instruct_SFT_mmstar_15_blind_inst  0.704453       NaN   0.331984   
Qwen3-VL-8B-Instruct_SFT_vqa_15_blind_inst     0.696356       NaN   0.307692   
humans                                              NaN       NaN   0.267739   
llava-v1.6-mistral-7b-hf                       0.421053       NaN   0.246964   
llava-v1.6-mistral-7b-hf_A3_vqa_15_blind_inst  0.425101       NaN   0.246964   

                                                count                    
                                              correct                    
condition                                              blind inst blind  
model                                                                    
InternVL3_5-1B                                  247.0  247.0      247.0  
InternVL3_5-2B                                  247.0  247.0      247.0  
InternVL3_5-4B                                  247.0  247.0      247.0  
InternVL3_5-8B                                  246.0  247.0      247.0  
InternVL3_5-8B_A1_vqa_gt                        247.0    NaN      247.0  
Qwen3-8B-Base                                   247.0    NaN        NaN  
Qwen3-VL-2B-Instruct                            247.0  247.0      247.0  
Qwen3-VL-4B-Instruct                            247.0  247.0      247.0  
Qwen3-VL-4B-Instruct_A1_vqa_gt                  247.0    NaN      247.0  
Qwen3-VL-4B-Instruct_A2_vqa_10_blind_inst       247.0    NaN      247.0  
Qwen3-VL-4B-Instruct_A3_vqa_15_blind_inst       247.0    NaN      247.0  
Qwen3-VL-4B-Instruct_A4_mmstar_15_blind_inst    247.0    NaN      247.0  
Qwen3-VL-4B-Instruct_SFT_vqa_15_blind_inst      247.0    NaN      247.0  
Qwen3-VL-4B-Instruct_SFT_vqa_gt                 247.0    NaN      247.0  
Qwen3-VL-8B-Instruct                            247.0  247.0      247.0  
Qwen3-VL-8B-Instruct_A1_vqa_gt                  247.0    NaN      247.0  
Qwen3-VL-8B-Instruct_A2_vqa_10_blind_inst       247.0    NaN      247.0  
Qwen3-VL-8B-Instruct_A3_vqa_15_blind_inst       247.0    NaN      247.0  
Qwen3-VL-8B-Instruct_A4_mmstar_15_blind_inst    247.0    NaN      247.0  
Qwen3-VL-8B-Instruct_SFT_mmstar_15_blind_inst   247.0    NaN      247.0  
Qwen3-VL-8B-Instruct_SFT_vqa_15_bl

In [66]:
human_vqa=pd.read_csv('/home/work/yuna/HPA/evaluation/scored/humans/human_vqa_per_question.csv')
# matching pids for human-model comparison 
human_vqa['model'] = "humans" 
human_vqa['condition'] = "inst blind" 

qids = human_vqa.qid.unique()
print(len(qids))
model_vqa = model_results['vqa_1k'] 

vqa_human_comparison = pd.concat([model_vqa[model_vqa['question_id'].isin(qids)] , human_vqa])
vqa_human_comparison.groupby(['model', 'condition']).mean(numeric_only=True)['correct'] 

374


model                                          condition        
InternVL3_5-1B                                 vqa 1k               0.804813
                                               vqa 1k blind         0.435829
                                               vqa 1k inst blind    0.438503
InternVL3_5-2B                                 vqa 1k               0.820856
                                               vqa 1k blind         0.455437
                                               vqa 1k inst blind    0.452763
InternVL3_5-4B                                 vqa 1k               0.829768
                                               vqa 1k blind         0.450089
                                               vqa 1k inst blind    0.472371
InternVL3_5-8B                                 vqa 1k               0.868984
                                               vqa 1k blind         0.471480
                                               vqa 1k inst blind    0.453654
InternVL3_5

In [ ]:
pt = vqa_human_comparison.pivot_table( 
    index=['model', 'folder'], 
    columns=['condition'], 
    values=['correct'],
    aggfunc=['mean', 'count']
)
# Round the "mean" rows/columns to 2 decimal places
pt = pt.round(4)
pt.to_csv(f'./vqa_human_comparison.csv')
pt 

In [ ]:
pt = df.pivot_table(
    index=['model', 'folder'], 
    columns=['condition'], 
    values=['correct'],
    aggfunc=['mean', 'count']
)
# Round the "mean" rows/columns to 2 decimal places
pt = pt.round(4)

# MMStar 

In [44]:
dfs = []
for filepath in glob("/home/work/yuna/HPA/results/swift/*mmstar*.jsonl"): 
    print(f"Evaluating: {filepath}")
    df = read_file(filepath) 
    dfs.append(df)
    # results = evaluate_results(filepath)
    # print_report(results)

Evaluating: /home/work/yuna/HPA/results/swift/llava-v1.6-mistral-7b-hf_mmstar.jsonl
Evaluating: /home/work/yuna/HPA/results/swift/InternVL3_5-4B_mmstar_blind.jsonl
'output' cannot process /home/work/yuna/HPA/results/swift/InternVL3_5-4B_mmstar_blind.jsonl
Evaluating: /home/work/yuna/HPA/results/swift/Qwen3-VL-4B-Instruct_mmstar_inst_blind.jsonl
Evaluating: /home/work/yuna/HPA/results/swift/InternVL3_5-4B_mmstar_inst_blind.jsonl
Evaluating: /home/work/yuna/HPA/results/swift/InternVL3_5-2B_mmstar_sys_inst_blind.jsonl
'output' cannot process /home/work/yuna/HPA/results/swift/InternVL3_5-2B_mmstar_sys_inst_blind.jsonl
Evaluating: /home/work/yuna/HPA/results/swift/llava-v1.6-mistral-7b-hf_mmstar_inst_blind.jsonl
Evaluating: /home/work/yuna/HPA/results/swift/Qwen3-VL-4B-Instruct_mmstar_blind.jsonl
Evaluating: /home/work/yuna/HPA/results/swift/InternVL3_5-4B_mmstar.jsonl
Evaluating: /home/work/yuna/HPA/results/swift/Qwen3-0.6B_mmstar.jsonl
Evaluating: /home/work/yuna/HPA/results/swift/llava-v

In [54]:
df = pd.concat(dfs)
df = df[df['condition'] != '_blind']
results = df.groupby(['model_full', 'condition', 'category', 'l2_category'])['correct'].mean().reset_index()
results.pivot_table(index=['model_full', 'condition'], columns=[ 'category', 'l2_category'], values=['correct'])

correct  \
category                                      coarse perception   
l2_category                                       image emotion   
model_full                        condition                       
OpenGVLab/InternVL3_5-2B          _inst_blind          0.193548   
OpenGVLab/InternVL3_5-4B                               0.000000   
                                  _inst_blind          0.258065   
OpenGVLab/InternVL3_5-8B                               0.774194   
                                  _inst_blind          0.451613   
Qwen/Qwen3-VL-4B-Instruct                              0.161290   
                                  _inst_blind          0.322581   
Qwen/Qwen3-VL-8B-Instruct                              0.483871   
                                  _inst_blind          0.161290   
llava-hf/llava-v1.6-mistral-7b-hf                      0.580645   
                                  _inst_blind          0.032258   

                                                                     \
category                                                              
l2_category                                   image scene and topic   
model_full                        condition                           
OpenGVLab/InternVL3_5-2B          _inst_blind              0.042553   
OpenGVLab/InternVL3_5-4B                                   0.047619   
                                  _inst_blind              0.063830   
OpenGVLab/InternVL3_5-8B                                   0.588652   
                                  _inst_blind              0.283688   
Qwen/Qwen3-VL-4B-Instruct                                  0.411348   
                                  _inst_blind              0.148936   
Qwen/Qwen3-VL-8B-Instruct                                  0.425532   
                                  _inst_blind              0.198582   
llava-hf/llava-v1.6-mistral-7b-hf                          0.453901   
                                  _inst_blind              0.212766   

                                                                     \
category                                                              
l2_category                                   image style & quality   
model_full                        condition                           
OpenGVLab/InternVL3_5-2B          _inst_blind              0.038462   
OpenGVLab/InternVL3_5-4B                                   0.333333   
                                  _inst_blind              0.000000   
OpenGVLab/InternVL3_5-8B                                   0.769231   
                                  _inst_blind              0.333333   
Qwen/Qwen3-VL-4B-Instruct                                  0.397436   
                                  _inst_blind              0.153846   
Qwen/Qwen3-VL-8B-Instruct                                  0.666667   
                                  _inst_blind              0.141026   
llava-hf/llava-v1.6-mistral-7b-hf                          0.628205   
                                  _inst_blind              0.089744   

                                                                       \
category                                      fine-grained perception   
l2_category                                              localization   
model_full                        condition                             
OpenGVLab/InternVL3_5-2B          _inst_blind                   0.000   
OpenGVLab/InternVL3_5-4B                                          NaN   
                                  _inst_blind                   0.100   
OpenGVLab/InternVL3_5-8B                                        0.675   
                                  _inst_blind                   0.250   
Qwen/Qwen3-VL-4B-Instruct                                       0.325   
                                  _inst_blind                   0.150   
Qwen/Qwen3-VL-8B-Instruct                                       0.550   
                                  _inst_bl